## **1. Setup and Configuration**


In [ ]:
import re

import urllib.request
import numpy as np
import pandas as pd
import os

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch.nn.functional as F

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
sns.set_style("white")

from scipy.stats import pearsonr

## **2. NSMC Training Data**

### **2.1 Load Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/Project/Sentiment_Analysis/klue_bert_model'

### **2.2 Clean Dataset**

In [ ]:
# Only load NSMC training data when model training is required

if not os.path.exists(MODEL_PATH):

  train_file = urllib.request.urlopen("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")

  train_data = pd.read_table(train_file)

  train_data.drop_duplicates(subset=["document"], inplace=True)

  train_data = train_data.dropna(how="any")

  train_data["document"] = train_data["document"].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", regex=True)

  train_data["document"] = train_data["document"].replace("", np.nan)

  train_data = train_data.dropna(how="any")

  print("NSMC training data loaded and cleaned")

else:

  print(
      "Saved model found - "
      "skipping NSMC training data loading"
  )

## **3. KLUE-BERT Sentiment Model**

### **3.1 Dataset Class**

In [ ]:
class NSMCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

### **3.2 Fine-Tuning**

In [ ]:
# Loading the KLUE-BERT Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH if os.path.exists(MODEL_PATH) else "klue/bert-base")

if os.path.exists(MODEL_PATH):
  # loading the saved model
  model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
  print("Saved model loaded")

else:
  train_dataset = NSMCDataset(train_data['document'].tolist(), train_data['label'].tolist(), tokenizer)

  model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2)

  # Setting up training arguments
  training_args = TrainingArguments(
      output_dir='./results',
      num_train_epochs=1,
      per_device_train_batch_size=32,
      gradient_accumulation_steps=1,
      learning_rate=2e-5,
      logging_steps=100,
      fp16=True if torch.cuda.is_available() else False
  )

  def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }

  # Initialize the Trainer and start training
  trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics
  )

  trainer.train()

  model.save_pretrained(MODEL_PATH)
  tokenizer.save_pretrained(MODEL_PATH)
  print("Training complete. Model saved", MODEL_PATH)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### **3.3 Long-Text Sentiment Inference**

In [ ]:
from src.preprocessing import (
    clean_nsmc_data,
    prepare_journal_data
)

from src.sentiment import (
    predict_bert_sentiment
)

from src.evaluation import (
    evaluate_mood_alignment,
    get_largest_discrepancies,
    print_alignment_metrics
)

## **4. Daily Reflection Data**

### **4.1 Load Dataset**

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()

gc = gspread.Client(auth=creds)

spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1DrDCLitpXZyJajcWYcMNcJCZU4Xa0aE_gj4sfXzl6x4/edit?gid=0#gid=0')

worksheet = spreadsheet.worksheet('data')

### **4.2 Clean and Format Data**

In [ ]:
df = pd.DataFrame(worksheet.get_all_records())

# Shortening Column Names
df = df.rename(columns={
    "Work Intensity (0 - Easy / 10 - Intense)": "Work Intensity",
    "Overall Mood (0 - Poor / 10 - Great)": "Overall Mood",
    "Name": "Date"
})

# Fixing Data Types
df["Hours of Work"] = pd.to_numeric(df["Hours of Work"], errors="coerce").fillna(0)
df["Work Intensity"] = pd.to_numeric(df["Work Intensity"], errors="coerce").fillna(0)
df["Overall Mood"] = pd.to_numeric(df["Overall Mood"], errors="coerce")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date')
df['Day_of_Week'] = df['Date'].dt.day_name()


df[[
      "Date",
      "Today's Status",
      "Hours of Work",
      "Work Intensity",
      "Overall Mood",
      "Day_of_Week"
    ]
].head()

### **4.3 Generate Daily Sentiment Score**

In [ ]:
df["Sentiment Score"] = df["Reflection"].apply(
    lambda text: predict_bert_sentiment(
        text=text,
        model=model,
        tokenizer=tokenizer,
        device=device,
        max_length=128
    )
)

df[[
      "Date",
      "Today's Status",
      "Hours of Work",
      "Work Intensity",
      "Overall Mood",
      "Day_of_Week",
      "Sentiment Score"
    ]
].head()

## **5. Model Validation Against Self-Reported Mood Score**

### **5.1 MAE / RMSE / Bias**

In [ ]:
df["Residual"] = (
    df["Sentiment Score"]
    - df["Overall Mood"]
)

df["Absolute Discrepancy"] = (
    df["Sentiment Score"]
    - df["Overall Mood"]
).abs()

mae = mean_absolute_error(
    df["Overall Mood"],
    df["Sentiment Score"]
)

rmse = np.sqrt(
    mean_squared_error(
        df["Overall Mood"],
        df["Sentiment Score"]
    )
)

mean_bias = df["Residual"].mean()

print(f"Mean Absolute Discrepancy (MAE): {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Mean Bias: {mean_bias:.3f}")

### **5.2 Pearson Correlation**

In [ ]:
# Calculate Pearson correlation between Sentiment Score and Overall Mood
# Note: pearsonr returns the correlation coefficient and the p-value.
correlation, p_value = pearsonr(
    df["Overall Mood"],
    df["Sentiment Score"]
)

print(f"Pearson Correlation: {correlation:.3f}")
print(f"P-value: {p_value:.4f}")

### **5.3 Largest Discrepancies**

In [ ]:
largest_discrepancies = (
    df[
        [
            "Date",
            "Overall Mood",
            "Sentiment Score",
            "Absolute Discrepancy"
        ]
    ]
    .sort_values(
        "Absolute Discrepancy",
        ascending=False
    )
    .head(10)
)

largest_discrepancies

### **5.4 Mood vs Sentiment Score Discrepancy Scatter**

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df["Overall Mood"],
    df["Sentiment Score"],
    alpha=0.7
)

plt.plot(
    [0, 10],
    [0, 10],
    linestyle="--",
    label="Perfect Agreement"
)

plt.xlim(0, 10)
plt.ylim(0, 10)

plt.xlabel("Self-Reported Overall Mood")
plt.ylabel("KLUE-BERT Sentiment Score")
plt.title("Self-Reported Mood vs KLUE-BERT Sentiment")

plt.legend()
plt.show()

## **6. Analysis**

### **6.1 Hours of Work vs Sentiment Score**

In [ ]:
on_duty = df[df["Today's Status"] == "On Duty"].copy()

In [ ]:
on_duty_filtered = on_duty[on_duty["Hours of Work"] > 0]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    on_duty_filtered["Hours of Work"],
    on_duty_filtered["Sentiment Score"],
    alpha=0.7
)

if len(on_duty_filtered) > 1:
    m, b = np.polyfit(on_duty_filtered["Hours of Work"], on_duty_filtered["Sentiment Score"], 1)

    x_line = np.linspace(on_duty_filtered["Hours of Work"].min(), on_duty_filtered["Hours of Work"].max(), 100)

    ax.plot(x_line, m * x_line + b, linestyle="--")

ax.axhline(5, linestyle=":", linewidth=1, alpha=0.5)

ax.set_title(
    "Hours of Work vs Sentiment\n"
    "(Excluding Days with 0 Hours of Work)"
)

ax.set_xlabel("Hours of Work")
ax.set_ylabel("Sentiment Score")
ax.set_ylim(0, 10)

plt.tight_layout()
plt.show()

In [ ]:
zero_work_days = on_duty[on_duty["Hours of Work"] == 0]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    np.random.uniform(
        -0.3,
        0.3,
        size=len(zero_work_days)
    ),
    zero_work_days["Sentiment Score"],
    alpha=0.7
)

ax.axhline(5, linestyle=":", linewidth=1, alpha=0.5)

ax.set_title(
    "Distribution of Sentiment Score\n"
    "(On-Duty Days with 0 Hours of Work)"
)

ax.set_ylabel("Sentiment Score")
ax.set_ylim(0, 10)
ax.set_xticks([])

plt.tight_layout()
plt.show()

### **6.2 Work Intensity vs Sentiment Score**

In [ ]:
on_duty_intensity = on_duty[on_duty['Work Intensity'] > 0]  # Filter out zero-intensity entries

fig, ax3 = plt.subplots(figsize=(8, 6))
ax3.scatter(on_duty_intensity['Work Intensity'], on_duty_intensity['Sentiment Score'],
            alpha=0.7, edgecolors='white', linewidths=0.5)
if len(on_duty_intensity) > 1:
    m2, b2 = np.polyfit(on_duty_intensity['Work Intensity'], on_duty_intensity['Sentiment Score'], 1)
    x_line2 = np.linspace(on_duty_intensity['Work Intensity'].min(), on_duty_intensity['Work Intensity'].max(), 100)
    ax3.plot(x_line2, m2 * x_line2 + b2, color='gray', linestyle='--', linewidth=1.2)
ax3.set_ylim(0, 10)
ax3.axhline(5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax3.set_title('Work Intensity vs Sentiment\n(Excluding Days with 0 Work Intensity)', fontsize=13)
ax3.set_xlabel('Work Intensity')
ax3.set_ylabel('Sentiment Score')

plt.tight_layout()
plt.show()